# Zomato Delivery Operations Analysis Using PySpark
## Stage 5: Delivery-Time Prediction Using Spark MLlib Pipelines

**Author:** Computer Engineering Mini-Project  
**Framework:** Apache Spark MLlib (`pyspark.ml`)  
**Target Variable:** `Time_taken (min)` (Delivery Duration in Minutes)

---

### Objectives of Stage 5:
1. Formulate a multi-factor regression problem to estimate delivery duration using operational features.
2. Prevent data leakage by excluding identifiers (`ID`, `Delivery_person_ID`) and fitting feature transformations strictly on the training set.
3. Assemble a production Spark MLlib pipeline containing `Imputer`, `StringIndexer`, `OneHotEncoder`, and `VectorAssembler`.
4. Establish a naive baseline (predicting the training-set median delivery time).
5. Train and compare two candidate models: **Linear Regression** and **Random Forest Regressor**.
6. Evaluate performance using **RMSE**, **MAE**, and **R²**.
7. Extract and interpret feature importance weights.

In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from src.data_loader import get_spark_session
from src.prediction import train_delivery_prediction_models

spark = get_spark_session('Stage5_ML_Notebook')
parquet_path = '../data/processed/zomato_cleaned.parquet'
df = spark.read.parquet(parquet_path)
print(f'Loaded {df.count():,} records for machine learning modeling.')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/13 23:52:28 WARN Utils: Your hostname, Rajdeeps-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.35 instead (on interface en0)
26/10/13 23:52:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/13 23:52:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Loaded 45,584 records for machine learning modeling.


### Step 1: Feature Architecture & Pipeline Construction
We define our numerical, imputable, and categorical features:
- **Numerical:** Courier Age, Courier Ratings, Vehicle Condition, Multiple Deliveries, Is_Weekend
- **Imputable:** Straight-Line Distance (`Approx_Distance_Km`), Order Hour (`Order_Hour`)
- **Categorical:** Weather Conditions, Traffic Density, Order Type, Vehicle Type, Festival, City, Day of Week

In [2]:
# Run full MLlib training, evaluation, and pipeline export
results = train_delivery_prediction_models(
    df,
    train_ratio=0.8,
    seed=42,
    model_save_dir='../outputs/models/rf_pipeline_model'
)

STAGE 5: DELIVERY-TIME PREDICTION USING SPARK MLLIB

[Step 1] Splitting data: 80% Train / 19% Test (seed=42)


 Train records: 36,570 | Test records: 9,014

[Step 2] Evaluating Baseline Model (Predicts Training Median)...


 * Baseline (Median=25.0 min): RMSE = 9.522 min | MAE = 7.594 min

[Step 3] Fitting Linear Regression Model...


 * Linear Regression: RMSE = 6.034 min | MAE = 4.790 min | R2 = 0.590

[Step 4] Fitting Random Forest Regressor (30 trees, maxDepth=8)...


 * Random Forest:     RMSE = 4.793 min | MAE = 3.759 min | R2 = 0.741

[Step 5] Extracted 41 feature importance weights from Random Forest.

[Step 6] Saving best model pipeline to: ../outputs/models/rf_pipeline_model


 Random Forest PipelineModel persisted successfully.

STAGE 5 MODELING COMPLETED SUCCESSFULLY!


### Step 2: Model Performance Comparison
We compare the models across Root Mean Squared Error (RMSE) and Mean Absolute Error (MAE):

In [3]:
import pandas as pd

metrics_table = pd.DataFrame([
    {
        'Model': 'Baseline (Train Median = 25 min)',
        'RMSE (min)': results['baseline']['rmse'],
        'MAE (min)': results['baseline']['mae'],
        'R² Score': 'N/A',
        'Description': 'Naive operational benchmark'
    },
    {
        'Model': 'Linear Regression (Regularized)',
        'RMSE (min)': results['linear_regression']['rmse'],
        'MAE (min)': results['linear_regression']['mae'],
        'R² Score': results['linear_regression']['r2'],
        'Description': 'Linear parametric baseline'
    },
    {
        'Model': 'Random Forest Regressor (30 trees, depth 8)',
        'RMSE (min)': results['random_forest']['rmse'],
        'MAE (min)': results['random_forest']['mae'],
        'R² Score': results['random_forest']['r2'],
        'Description': 'Non-linear tree ensemble'
    }
])

print(metrics_table.to_string(index=False))

                                      Model  RMSE (min)  MAE (min) R² Score                 Description
           Baseline (Train Median = 25 min)       9.522      7.594      N/A Naive operational benchmark
            Linear Regression (Regularized)       6.034      4.790     0.59  Linear parametric baseline
Random Forest Regressor (30 trees, depth 8)       4.793      3.759    0.741    Non-linear tree ensemble


### Step 3: Operational Discussion & Real-World Generalization
1. **Non-Linear Interactions:** The Random Forest model achieved a **49.7% reduction in RMSE** over the median baseline, lowering MAE to **3.76 minutes** on unseen data.
2. **Predictive Association vs. Causality:** A model predicting longer delivery times during jams or rain reflects empirical correlation under existing operational practices, not strict causal proof.
3. **Missing Features:** Real-world logistics durations also depend on restaurant kitchen backlogs, elevator wait times in high-rises, and actual turn-by-turn road topology, which are absent from this observational dataset.

### Step 4: Shutdown Spark Session

In [4]:
spark.stop()
print('SparkSession stopped.')

SparkSession stopped.
